# SmartScan — train the occupancy predictor on Kaggle

**SIH 26055 · Smart Scan Strategy for Electronic Warfare**

Trains the next-slot occupancy predictor (GRU / dilated TCN / Transformer) with
privileged-teacher distillation, then exports the best model to **ONNX opset 17**
for embedded deployment.

### Before you run
1. **Add data** → search `ew-smart-scan-rf-environment` → Add.
2. **Accelerator: T4.** The pushed metadata requests `machine_shape:
   NvidiaTeslaT4`, so this is already set. Do *not* switch to P100 -- it is
   compute capability sm_60 and Kaggle's torch build supports sm_70+, so the
   GPU would be present but unusable and the run falls back to CPU.
3. Run all. About 25 minutes on a T4.

### Two things this notebook is careful about
* **It streams the dataset.** Attached datasets are mounted read-only at
  `/kaggle/input` and do *not* count against the 20 GB writable disk — but
  copying one there would. Nothing is copied.
* **No credential is ever printed.** There is nothing to authenticate for
  reading; publishing at the end uses Kaggle's own session, and the cell that
  does it prints no token.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('./kaggle_working')
WORK.mkdir(parents=True, exist_ok=True)
print('running on Kaggle' if ON_KAGGLE else 'running locally')

# The package itself is small; installing from source keeps the notebook and the
# repository in lockstep rather than pinned to a stale wheel.
if ON_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/shirish-raj-gupta/SIH26055_Prototype.git'], check=False)
try:
    import smartscan; print('smartscan', smartscan.__version__)
except ImportError:
    sys.path.insert(0, '..' if not ON_KAGGLE else '/kaggle/input/smartscan-source')
    import smartscan; print('smartscan (from source)', smartscan.__version__)

In [ ]:
import torch

# `torch.cuda.is_available()` is NOT sufficient on Kaggle. The accelerator you
# get is assigned, not chosen -- the API exposes only `enable_gpu`, with no way
# to request a type -- and a Tesla P100 is compute capability sm_60, which
# Kaggle's own torch build (cu128, sm_70+) cannot run. is_available() still
# returns True there; the failure only appears when a kernel is launched.
# So launch one and see.
GPU_USABLE = False
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {name}  sm_{major}{minor}  ({mem:.1f} GB)  x{torch.cuda.device_count()}')
    try:
        (torch.zeros(8, 8, device='cuda') @ torch.zeros(8, 8, device='cuda')).cpu()
        GPU_USABLE = True
    except Exception as exc:
        print(f'  GPU present but UNUSABLE by this torch build: {type(exc).__name__}')
        print(f'  {str(exc).splitlines()[0][:160]}')
        print('  -> falling back to CPU. Re-run with Accelerator = GPU T4 x2 for the')
        print('     full three-architecture comparison.')
else:
    print('no GPU allocated')

DEVICE = 'cuda' if GPU_USABLE else 'cpu'
print(f'device: {DEVICE}')
print('torch', torch.__version__)


## Locate the dataset

`load_dataset` tries, in order: an explicit path, the local cache, Kaggle, and
finally **regeneration from seeds**. On Kaggle the attached copy is found
immediately; the fallback exists so the same notebook runs offline.

In [ ]:
from smartscan.data.kaggle_io import load_dataset

OWNER = 'shirishrajgupta'   # the dataset's owner; the library default is not this

# Search recursively and report. The previous one-level glob found nothing and
# the code then fell through to downloading under a DIFFERENT owner, failed,
# and silently regenerated episodes from seeds -- training on a corpus that was
# not the published one. A wrong-but-plausible corpus is worse than a crash.
ROOT = None
if ON_KAGGLE:
    mounts = sorted(Path('/kaggle/input').glob('*'))
    print('mounted at /kaggle/input:', [m.name for m in mounts] or '(nothing attached)')
    hit = next(Path('/kaggle/input').rglob('index.parquet'), None)
    if hit is None:
        print('No index.parquet under /kaggle/input.')
        print('Attach it: Add Input -> Datasets -> ew-smart-scan-rf-environment')
        print('Refusing to regenerate from seeds -- that would silently train on')
        print('a different corpus than the results claim.')
        raise SystemExit('dataset not attached')
    ROOT = hit.parent
    print('dataset root:', ROOT)

train_ds = load_dataset('train', root=ROOT, owner=OWNER)
val_ds   = load_dataset('val',   root=ROOT, owner=OWNER)
print(f'source     {train_ds.source}')
print(f'train      {len(train_ds)} episodes')
print(f'val        {len(val_ds)} episodes')
assert train_ds.source != 'regenerated', 'fell back to regeneration; see above'

# Splits are assigned BY SEED, so no episode can appear in both.
overlap = set(train_ds.episode_ids()) & set(val_ds.episode_ids())
assert not overlap, f'LEAKAGE: {len(overlap)} episodes in both splits'
print('no train/val overlap OK')


In [ ]:
import shutil

# The 20 GB limit applies to what we WRITE, not what we read.
free_gb = shutil.disk_usage(WORK).free / 1024**3
print(f'{free_gb:.1f} GB free in {WORK} (writable)')
if ROOT is not None:
    size = sum(f.stat().st_size for f in ROOT.rglob('*') if f.is_file()) / 1024**3
    print(f'{size:.2f} GB attached read-only at {ROOT} — streamed, never copied')

## Build windows

Only the **open-loop baseline** traces are used. Their coverage is close to
uniform, so the training distribution is not biased toward the behaviour of the
policy the predictor is later meant to improve on. The closed-loop traces in the
dataset are there for offline policy evaluation, not for this.

In [ ]:
from smartscan.config import load_config
from smartscan.data.kaggle_io import OccupancyWindowDataset

cfg = load_config('medium.yaml')
N_TRAIN_EPISODES = 120   # raise for a longer run; each adds ~250 windows
N_VAL_EPISODES   = 30

train_ds.index = train_ds.index.head(N_TRAIN_EPISODES)
val_ds.index   = val_ds.index.head(N_VAL_EPISODES)

train_windows = OccupancyWindowDataset(
    train_ds, window=cfg.predictor.window_slots, stride=16,
    agent='sequential', max_windows_per_episode=256, class_balanced=True,
)
val_windows = OccupancyWindowDataset(
    val_ds, window=cfg.predictor.window_slots, stride=64,
    agent='sequential', max_windows_per_episode=64,
)
print(f'train windows {len(train_windows)}')
print(f'val windows   {len(val_windows)}')

x, y, mask, y_true = train_windows[0]
print(f'x {tuple(x.shape)}  (planes: visit, hit, reported SNR, staleness)')
print(f'y {tuple(y.shape)}  mask covers {int(mask.sum())} channels — the tuned window')
print(f'positive rate in the privileged label: {float(y_true.mean()):.3%}')

## Train, with privileged distillation

The **teacher** sees the full occupancy tensor. The **student** sees only the
observation history, plus a KL term to the teacher over *all* channels — so the
teacher supplies soft labels exactly where the student has none.

This is training-time only. `PrivilegedAccess` refuses to open in evaluation
mode, so a deployed student cannot read ground truth even by accident.

In [ ]:
from smartscan.agents.predictors import build_predictor, masked_focal_loss


def train_one(arch, epochs=8, lr=3e-4, lambda_kd=0.5, temperature=2.0,
              teacher_epochs=4, patience=6, batch_size=64):
    """Train one architecture; returns the **best-val** student, not the last.

    Without restoring the best epoch, the `best val` printed below would
    describe a model that no longer exists. That is not academic here: this
    predictor overfits early -- on a 16-episode CPU corpus the validation
    loss bottomed at epoch 1 (0.0268) and then rose for six straight epochs
    while training loss kept falling.
    """
    import copy
    torch.manual_seed(cfg.run.seed)
    train_loader = train_windows.loader(batch_size=batch_size, seed=cfg.run.seed, num_workers=2)
    val_loader   = val_windows.loader(batch_size=batch_size, num_workers=2)
    hist = {'arch': arch, 'train': [], 'val': []}

    teacher = build_predictor(cfg, arch).to(DEVICE)
    opt_t = torch.optim.Adam(teacher.parameters(), lr=lr)
    for ep in range(teacher_epochs):
        teacher.train(); tot = n = 0
        for xb, _yb, _mb, ytb in train_loader:
            xb, ytb = xb.to(DEVICE), ytb.to(DEVICE)
            full = torch.ones_like(ytb, dtype=torch.bool)
            loss = masked_focal_loss(teacher(xb), ytb, full,
                                     cfg.predictor.focal_gamma, cfg.predictor.focal_alpha)
            opt_t.zero_grad(); loss.backward(); opt_t.step()
            tot += float(loss.detach()); n += 1
        print(f'  [{arch}] teacher {ep+1}/{teacher_epochs} loss={tot/max(n,1):.4f}', flush=True)
    teacher.eval()

    student = build_predictor(cfg, arch).to(DEVICE)
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    best_val, best_epoch = float('inf'), 0
    best_state = copy.deepcopy(student.state_dict())
    for ep in range(epochs):
        student.train(); tot = n = 0
        for xb, yb, mb, _ytb in train_loader:
            xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
            logits = student(xb)
            loss = masked_focal_loss(logits, yb, mb,
                                     cfg.predictor.focal_gamma, cfg.predictor.focal_alpha)
            with torch.no_grad():
                soft = torch.sigmoid(teacher(xb) / temperature)
            kd = torch.nn.functional.binary_cross_entropy_with_logits(
                logits / temperature, soft)
            loss = loss + lambda_kd * (temperature ** 2) * kd
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss.detach()); n += 1

        student.eval(); vt = vn = 0
        with torch.no_grad():
            for xb, yb, mb, _ytb in val_loader:
                vt += float(masked_focal_loss(student(xb.to(DEVICE)), yb.to(DEVICE), mb.to(DEVICE),
                                              cfg.predictor.focal_gamma, cfg.predictor.focal_alpha))
                vn += 1
        hist['train'].append(tot/max(n,1)); hist['val'].append(vt/max(vn,1))
        improved = hist['val'][-1] < best_val - 1e-6
        if improved:
            best_val, best_epoch = hist['val'][-1], ep + 1
            best_state = copy.deepcopy(student.state_dict())
        print(f"  [{arch}] student {ep+1}/{epochs} train={hist['train'][-1]:.4f} "
              f"val={hist['val'][-1]:.4f}{' *' if improved else ''}", flush=True)
        if patience and ep + 1 - best_epoch >= patience:
            print(f'  [{arch}] early stop: no val gain in {patience} epochs '
                  f'(best {best_val:.4f} @ epoch {best_epoch})', flush=True)
            break

    student.load_state_dict(best_state)   # the artefact now matches the number
    hist['best_epoch'] = best_epoch
    hist['best_val'] = best_val
    return student, hist

In [ ]:
# gru and tcn walk 128 time steps sequentially; on CPU that is ~7.8 s and
# ~7.2 s per batch against the transformer's 0.36 s, i.e. hours per epoch.
# Without a usable GPU, run only the architecture that can actually finish
# and say so, rather than reporting a comparison that never ran.
ARCHS = ('gru', 'tcn', 'transformer') if GPU_USABLE else ('transformer',)
if not GPU_USABLE:
    print('CPU run: training transformer only (gru/tcn need a GPU)')

# `batch_size` is not the number of sequences. Every model folds the 128
# channels into the batch dimension, so batch 64 is really 64*128 = 8192
# sequences of 128 steps -- and a 2-layer GRU over that asked a 16 GB T4 for
# a single 6.52 GiB allocation and died. The right batch depends on the GPU
# you were assigned, which you do not control, so find it rather than assume
# it: halve on OOM and record what actually ran.
BATCH0 = 64
MIN_BATCH = 4

results = {}
t0 = time.time()
for arch in ARCHS:
    print(f'=== {arch} ===', flush=True)
    bs = BATCH0
    while True:
        try:
            model, hist = train_one(arch, batch_size=bs)
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs <= MIN_BATCH:
                raise
            bs //= 2
            print(f'  CUDA OOM -> retrying at batch {bs} '
                  f'({bs * cfg.n_channels} sequences)', flush=True)
    hist['batch_size'] = bs
    n_params = sum(p.numel() for p in model.parameters())
    results[arch] = {'model': model, 'history': hist, 'params': n_params}
    print(f"  {n_params:,} parameters, batch {bs}, best val {hist['best_val']:.4f} "
          f"@ epoch {hist['best_epoch']} of {len(hist['val'])} run "
          f"(weights restored to that epoch)", flush=True)
    print(flush=True)
print(f'total {time.time()-t0:.0f}s')


## Score against the privileged label

The student never *sees* ground truth, but it is scored against it: the question
is how well it predicts the whole band, including the 31/32 of channels the
receiver could not observe.

In [ ]:
import numpy as np

from smartscan.analysis.metrics import prediction_scores

scores = {}
val_loader = val_windows.loader(batch_size=128)
for arch, r in results.items():
    probs, truths = [], []
    r['model'].eval()
    with torch.no_grad():
        for xb, _yb, _mb, ytb in val_loader:
            probs.append(torch.sigmoid(r['model'](xb.to(DEVICE))).cpu().numpy())
            truths.append(ytb.numpy())
    scores[arch] = prediction_scores(np.concatenate(truths), np.concatenate(probs))
    scores[arch]['params'] = r['params']

print(f"{'arch':>14}{'params':>10}{'AUC':>8}{'F1':>8}{'recall':>9}{'Brier':>9}")
for arch, s in scores.items():
    print(f"{arch:>14}{s['params']:10,}{s['auc']:8.3f}{s['f1']:8.3f}"
          f"{s['recall']:9.3f}{s['brier']:9.4f}")

BEST = max(scores, key=lambda a: scores[a]['auc'])
print(f'\nbest by AUC: {BEST}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.2))
for arch, r in results.items():
    ax.plot(r['history']['val'], marker='o', ms=3, label=f"{arch} (val)")
    ax.plot(r['history']['train'], ls='--', alpha=0.5, label=f"{arch} (train)")
ax.set_xlabel('epoch'); ax.set_ylabel('masked focal loss')
ax.set_title('Predictor training'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(WORK / 'predictor_curves.png', dpi=150)
plt.show()

## Export to ONNX (opset 17)

This is the artefact that goes to embedded hardware. Opset 17 is pinned because
it is what current ONNX Runtime builds for ARM support without custom operators.

In [ ]:
best_model = results[BEST]['model'].cpu().eval()
dummy = torch.zeros(1, 4, cfg.n_channels, cfg.predictor.window_slots)
onnx_path = WORK / f'predictor_{BEST}_opset17.onnx'

torch.onnx.export(
    best_model, dummy, str(onnx_path),
    opset_version=17,
    input_names=['observation_window'], output_names=['occupancy_logits'],
    dynamic_axes={'observation_window': {0: 'batch'}, 'occupancy_logits': {0: 'batch'}},
)
print(f'wrote {onnx_path} ({onnx_path.stat().st_size/1024:.0f} KB)')

# Verify the export actually agrees with torch, rather than assuming it does.
try:
    import onnx
    import onnxruntime as ort
    onnx.checker.check_model(onnx.load(str(onnx_path)))
    sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    onnx_out = sess.run(None, {'observation_window': dummy.numpy()})[0]
    with torch.no_grad():
        torch_out = best_model(dummy).numpy()
    delta = float(np.abs(onnx_out - torch_out).max())
    print(f'ONNX vs torch max abs difference: {delta:.2e}')
    assert delta < 1e-4, 'ONNX export disagrees with the torch model'
    print('ONNX export verified ✓')
except ImportError:
    print('onnx/onnxruntime not installed — export written but NOT verified')

In [ ]:
# Checkpoints and metrics, for publication as the `ew-smart-scan-models` dataset.
for arch, r in results.items():
    torch.save(r['model'].cpu().state_dict(), WORK / f'predictor_{arch}.pt')

metrics = {
    'task': 'next-slot occupancy prediction',
    'best_arch': BEST,
    'scores': {a: {k: float(v) for k, v in s.items()} for a, s in scores.items()},
    'histories': {a: r['history'] for a, r in results.items()},
    'config_hash': cfg.hash(),
    'n_train_episodes': len(train_ds),
    'n_val_episodes': len(val_ds),
    'device': DEVICE,
    'distillation': 'privileged teacher, training-time only',
}
(WORK / 'predictor_metrics.json').write_text(json.dumps(metrics, indent=2))

print('artefacts in', WORK)
for f in sorted(WORK.iterdir()):
    print(f'  {f.name:34} {f.stat().st_size/1024:8.1f} KB')

## Publish the weights

Everything in `/kaggle/working` is saved with the notebook version. To turn it
into the reusable `ew-smart-scan-models` dataset, use **File → Save Version**,
then *New Dataset* from the output — or run the cell below, which uses the
session's own credentials and prints none of them.

Inference then needs one line:
`kagglehub.dataset_download('<user>/ew-smart-scan-models')`.

In [ ]:
# Credentials come from Kaggle Secrets, never from this file.
#
# This notebook is PUBLIC. Anything pasted here -- a .env line, a token, a
# username and key -- is published with it and is indexed. Training needs no
# credentials at all: the dataset arrives through `dataset_sources` and the
# package installs from a public repo. Only this optional publish step needs
# auth, so it reads Kaggle Secrets at run time.
#
# To enable:  Add-ons -> Secrets -> attach KAGGLE_USERNAME and KAGGLE_KEY
# (from https://www.kaggle.com/settings/api). Secrets are bound to the
# session, not written into the notebook.
PUBLISH = True    # KAGGLE_USERNAME / KAGGLE_KEY attached as Secrets

def _secret(name):
    """Return a Kaggle Secret, or None. Never prints or returns a value."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

if PUBLISH:
    user, key = _secret('KAGGLE_USERNAME'), _secret('KAGGLE_KEY')
    if not (user and key):
        print('No KAGGLE_USERNAME / KAGGLE_KEY secret attached; skipping upload.')
        print('Attach them under Add-ons -> Secrets, or use File -> Save Version.')
    else:
        os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = user, key
        meta = {
            'title': 'EW Smart Scan: Trained Scheduler Models',
            'id': f'{user}/ew-smart-scan-models',
            'licenses': [{'name': 'CC-BY-SA-4.0'}],
        }
        (WORK / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
        r = subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'create',
                            '-p', str(WORK), '--dir-mode', 'zip', '--public'],
                           capture_output=True, text=True)
        # Print only the status: API output can echo request context.
        print('published' if r.returncode == 0 else f'failed (rc={r.returncode})')
else:
    print('PUBLISH is False - nothing uploaded. Use File -> Save Version instead.')
